In [1]:
import time
from pathlib import Path

import requests

from src.core import *
from src.gpt import *

In [2]:
np.random.seed(42)

In [3]:
class GPTEmbeddingWithPadding(GPTEmbedding):

    def forward(self, x: Tensor, positions):
        token = self.embedding(x)
        position = self.positional_embedding(Tensor(positions))
        return self.dropout(token + position)

In [4]:
class GPTAttentionWithPadding(GPTAttention):

    def forward(self, x: Tensor, valid):
        norm = self.normalize(x)
        head_embedding_size = self.embedding_size // self.heads
        multi_head_shape = (-1, x.shape[1], self.heads, head_embedding_size)
        shape = (-1, x.shape[1], self.embedding_size)

        query = self.query(norm).reshape(multi_head_shape).transpose((0, 2, 1, 3))
        key = self.key(norm).reshape(multi_head_shape).transpose((0, 2, 3, 1))
        value = self.value(norm).reshape(multi_head_shape).transpose((0, 2, 1, 3))
        scale = Tensor(np.array(1.0 / np.sqrt(head_embedding_size)))
        scores = self.mask(query @ key * scale)

        scores = Tensor(np.where(valid[:, None, None, :], scores.data, self.mask.value))
        weights = self.softmax(scores)
        return x + self.dropout(self.output((weights @ value).transpose((0, 2, 1, 3)).reshape(shape)))

In [5]:
class GPTTransformerWithPadding(Composite):

    def __init__(self, embedding_size, heads):
        self.attention = GPTAttentionWithPadding(embedding_size, heads)
        self.feed_forward = GPTFeedForward(embedding_size)

        super().__init__([self.attention,
                          self.feed_forward])

    def forward(self, x: Tensor, valid):
        return self.feed_forward(self.attention(x, valid))

In [6]:
class GPTWithPadding(Composite):

    def __init__(self, vocab_size, context_size, embedding_size, heads, blocks):
        self.embedding = GPTEmbeddingWithPadding(vocab_size, context_size, embedding_size)
        self.transformers = [GPTTransformerWithPadding(embedding_size, heads) for _ in range(blocks)]
        self.output = GPTOutput(embedding_size, vocab_size)

        super().__init__([self.embedding] + self.transformers + [self.output])

    def forward(self, x: Tensor, valid, positions):
        x = self.embedding(x, positions)
        for layer in self.transformers:
            x = layer(x, valid)
        return self.output(x)

In [7]:
class GPTBatchTrainer(GPTTrainer):

    def generate(self, dataset, prompts, steps=256):
        self.model.eval()
        batch_tokens = [dataset.encode(p) for p in prompts]
        n = len(batch_tokens)

        with Tensor.no_grad():
            for _ in range(steps):
                windows = [tokens[-dataset.context_size:] for tokens in batch_tokens]
                width = max(len(w) for w in windows)

                features = np.zeros((n, width), dtype=np.int64)
                valid = np.zeros((n, width), dtype=bool)
                positions = np.zeros((n, width), dtype=np.int64)
                for i, window in enumerate(windows):
                    pad = width - len(window)
                    features[i, pad:] = window
                    valid[i, pad:] = True
                    positions[i, pad:] = np.arange(len(window))

                logits = self.model(Tensor(features), valid, positions)

                for i, tokens in enumerate(batch_tokens):
                    last_logits = logits.data[i, -1]
                    exp = np.exp(last_logits - np.max(last_logits))
                    probs = exp / np.sum(exp)
                    token = np.random.choice(len(probs), p=probs)
                    tokens.append(token)
        return [dataset.decode(t) for t in batch_tokens]

In [8]:
LEARNING_RATE = 0.001

In [9]:
BATCH_SIZE = 16

In [10]:
CONTEXT_SIZE = 256

In [11]:
EMBEDDING_SIZE = 384

In [12]:
HEADS = 6

In [13]:
BLOCKS = 6

In [14]:
DATA_FILE = "../../../tinyshakespeare.txt"

file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

In [15]:
MODEL_FILE = "../../../tinyshakespeare-gpt.npz"

dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = GPTWithPadding(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
loss_fn = GPTLoss()
optimizer = AdamWOptimizer(model.parameters, lr=LEARNING_RATE)
trainer = GPTBatchTrainer(model, loss_fn, optimizer)
trainer.load(MODEL_FILE)

In [16]:
prompts = ["ROMEO:", "JULIET:", "HAMLET:", "MACBETH:"]

In [17]:
np.random.seed(42)

start = time.time()
for p in prompts:
    print(trainer.generate(dataset, prompts=[p])[0])
print(f"{time.time() - start:.8f} seconds")

ROMEO:
When I no woman me, I have negless'd thee
Which customs can do him:
You sir.

BUCKINGHAM:
There in English be, whose thou begin
Of abused wonder than he was done.

QUEEN ELIZABETH:
Wilt thou wilt to salt Cap, in direction
Making the dom office whose his l
JULIET:
Since my noble true-lost hate, but that I bear to do leave.

CLAUDIO:
No, pardon, I learn!

LEONTES:
What ourself God france?

Nurse:
I thought were if I would fortune men a soldier.

WARWICK:
These Duke of York yet usue where it is.

GLOUCESTER:
Though h
HAMLET:
We have hour had tyrrown kiss speak.

QUEEN ELIZABETH:
Where your faults air appetite Marcius bought
With the king, and never haste upon blood
He's wife came the wars: being for mine,
She be avoiden of you: yea, this consent?

KING RICHARD IIII:
The queen
MACBETH:
Welcome, sir, fend my lord? Why, no; withou would he
diest the childge Tumber with burn speedful childite
As I do bold a stranger again. Then, marquoth;
Will not turn at the sea tomb;
Where sconder'd 

In [18]:
np.random.seed(42)

start = time.time()
results = trainer.generate(dataset, prompts=prompts)
for r in results:
    print(r)
print(f"{time.time() - start:.8f} seconds")

ROMEO:
Consonscience me I through you again and all good narly to yet:
Stay, if you all.

ROMEO:
We say, they, say, I'll business one: but
your lates, he do before than you came; for
ever come you all the world, my safe I hear thee in honour!
My bosom shame, who
JULIET:
Ay, as you do issue more hath mine against meQuess
Being may, but his and my went.

KING RICHARD IIII:
How brow, his time; I could them so
Consure learn her leaved with going your shepherd.

QUEEN ELIZABETH:
Hope, love, and God will you unkel.

KING RICHA
HAMLET:
A citizen husband.

NORFOLK:
How! is your love friar.

YORK:
I may not a night of I concluse to you.

Nurse:
It gentlewoman?
He would faith me, if you as fire come,
Give no fear'd change by thee ne'er-woman!
Ah, and if I shall intege thee evide
To with vo
MACBETH:
Worthy thy suffer; I shall sker home for Gloucester.
Go, she, man: is never an even imprisonation?
Have my parties try perful keep,--
Could know do I how now my kinders?

First Servingman:
Thou didst 